# Session 6 — Auth & Security
## Empire & Ink: AI Mughal Art Studio

**What you'll build:** A complete authentication system using Supabase Auth. Users register, log in, get a JWT, and FastAPI validates it on every request via Depends().

---

## Setup

In [ ]:
!pip install fastapi httpx supabase python-dotenv

## Theory 1 — Why Authentication Matters

Without auth, anyone on the internet can:
- Generate thousands of images on your Gemini quota
- Read everyone else's gallery
- Delete other users' data

**Authentication** proves *who you are*. **Authorisation** controls *what you can do*.

In [ ]:
users_db = {}

def register(email, password):
    if email in users_db:
        return {"error": "Already registered"}
    users_db[email] = password
    return {"message": "Registered", "email": email}

def login(email, password):
    if users_db.get(email) != password:
        return {"error": "Invalid credentials"}
    return {"access_token": f"fake-jwt-for-{email}"}

print(register("artist@example.com", "secret123"))
print(login("artist@example.com", "secret123"))
print(login("artist@example.com", "wrongpass"))

## Theory 2 — JWT Structure

A **JWT** (JSON Web Token) has three dot-separated parts:
```
header.payload.signature
```
- **Header**: algorithm used (HS256)
- **Payload**: claims (user id, email, expiry timestamp)
- **Signature**: cryptographic proof it hasn't been tampered with

Visit [jwt.io](https://jwt.io) to inspect tokens visually. Never trust client-side decoding for security decisions.

In [ ]:
import base64, json

def decode_jwt_payload(token: str) -> dict:
    try:
        parts = token.split(".")
        if len(parts) != 3:
            return {"error": "Not a valid JWT"}
        payload_bytes = base64.urlsafe_b64decode(parts[1] + "==")
        return json.loads(payload_bytes)
    except Exception as e:
        return {"error": str(e)}

sample_jwt = (
    "eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9"
    ".eyJzdWIiOiIxMjM0NTY3ODkwIiwibmFtZSI6IkFydGlzdCIsImlhdCI6MTUxNjIzOTAyMn0"
    ".SflKxwRJSMeKKF2QT4fwpMeJf36POk6yJV_adQssw5c"
)
print(decode_jwt_payload(sample_jwt))

## Theory 3 — FastAPI Depends() & Bearer Tokens

`Depends()` is FastAPI's dependency injection. Define once, inject everywhere:

```python
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
security = HTTPBearer()

async def get_current_user(credentials: HTTPAuthorizationCredentials = Depends(security)):
    token = credentials.credentials
    user = supabase.auth.get_user(token)
    return user

@app.get("/api/gallery")
def get_gallery(current_user = Depends(get_current_user)):
    return {"user_id": current_user.id}
```

In [ ]:
def extract_bearer_token(auth_header: str) -> str:
    if not auth_header or not auth_header.startswith("Bearer "):
        raise ValueError("Missing or invalid Authorization header")
    return auth_header.replace("Bearer ", "")

def verify_fake_token(token: str) -> dict:
    if token.startswith("valid-token-"):
        user_id = token.replace("valid-token-", "")
        return {"id": user_id, "email": f"{user_id}@example.com"}
    raise ValueError("Invalid token")

header = "Bearer valid-token-user123"
token  = extract_bearer_token(header)
user   = verify_fake_token(token)
print("Authenticated user:", user)

---
## In-Class Exercises

### Exercise 1 — sign_up() and sign_in() with Supabase Auth

In [ ]:
import os
try:
    from google.colab import userdata
    SUPABASE_URL = userdata.get("SUPABASE_URL")
    SUPABASE_KEY = userdata.get("SUPABASE_KEY")
except Exception:
    SUPABASE_URL = os.getenv("SUPABASE_URL", "")
    SUPABASE_KEY = os.getenv("SUPABASE_KEY", "")

def sign_up(supabase_client, email: str, password: str) -> dict:
    try:
        # result = supabase_client.auth.sign_up({"email": email, "password": password})
        return {"user": {"email": email}, "session": {"access_token": "placeholder-token"}}
    except Exception as e:
        return {"error": str(e)}

def sign_in(supabase_client, email: str, password: str) -> dict:
    try:
        # result = supabase_client.auth.sign_in_with_password({"email": email, "password": password})
        return {"user": {"email": email}, "session": {"access_token": "placeholder-token"}}
    except Exception as e:
        return {"error": str(e)}

print(sign_up(None, "test@example.com", "password123"))
print(sign_in(None, "test@example.com", "password123"))

### Exercise 2 — sign_out()

In [ ]:
def sign_out(supabase_client, access_token: str) -> bool:
    try:
        # supabase_client.auth.sign_out()
        return True
    except Exception:
        return False

print("Sign out:", sign_out(None, "placeholder-token"))

### Exercise 3 — get_current_user() dependency

In [ ]:
from fastapi import FastAPI, Depends, HTTPException, status
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials

app = FastAPI()
security = HTTPBearer()

def get_current_user(credentials: HTTPAuthorizationCredentials = Depends(security)):
    token = credentials.credentials
    # YOUR CODE HERE — in real code: supabase.auth.get_user(token)
    # Simulate: valid tokens start with "valid-"
    if token.startswith("valid-"):
        return {"id": "user-123", "email": "artist@example.com"}
    raise HTTPException(status_code=status.HTTP_401_UNAUTHORIZED, detail="Invalid token")

print("get_current_user dependency defined")

### Exercise 4 — Protect a route with Depends

In [ ]:
from fastapi.testclient import TestClient

@app.get("/api/me")
def get_me(current_user=Depends(get_current_user)):
    # YOUR CODE HERE — return user_id and email from current_user
    pass

client = TestClient(app)
r = client.get("/api/me", headers={"Authorization": "Bearer valid-user123"})
print("With valid token:", r.status_code, r.json())

### Exercise 5 — Test with and without token

In [ ]:
r1 = client.get("/api/me", headers={"Authorization": "Bearer valid-user123"})
print(f"Valid token  -> {r1.status_code}: {r1.json()}")

r2 = client.get("/api/me", headers={"Authorization": "Bearer bad-token"})
print(f"Bad token    -> {r2.status_code}: {r2.json()}")

r3 = client.get("/api/me")
print(f"No token     -> {r3.status_code}: {r3.json()}")

---
## Post-Class Exercises

### Challenge 1 — AuthResult dataclass

In [ ]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class AuthResult:
    success: bool
    user_id: Optional[str] = None
    email: Optional[str] = None
    access_token: Optional[str] = None
    error: Optional[str] = None

    # YOUR CODE HERE — add is_authenticated property
    @property
    def is_authenticated(self) -> bool:
        pass

ok   = AuthResult(success=True, user_id="u1", email="a@b.com", access_token="tok")
fail = AuthResult(success=False, error="Wrong password")
print("OK authenticated:  ", ok.is_authenticated)
print("Fail authenticated:", fail.is_authenticated)

### Challenge 2 — Token expiry handling

In [ ]:
import time

def is_token_expired(token: str) -> bool:
    payload = decode_jwt_payload(token)
    if "exp" not in payload:
        return True  # no expiry claim = treat as expired
    return payload["exp"] < time.time()

print("Sample JWT expired:", is_token_expired(sample_jwt))

### Challenge 3 — register → login → verify test

In [ ]:
import time

def test_auth_flow(supabase_client):
    email    = f"test-{int(time.time())}@example.com"
    password = "TestPassword123!"
    print("1. Registering...")
    reg = sign_up(supabase_client, email, password)
    print("   Result:", reg.get("user", reg.get("error")))
    print("2. Logging in...")
    result = sign_in(supabase_client, email, password)
    token = result.get("session", {}).get("access_token")
    print("   Token (first 20):", token[:20] if token else "None")
    print("3. Verifying expiry...")
    print("   Expired:", is_token_expired(token) if token else True)
    print("4. Signing out...")
    print("   Signed out:", sign_out(supabase_client, token))

test_auth_flow(None)

### Challenge 4 — Wrong password test

In [ ]:
def wrong_password_test(supabase_client):
    result = sign_in(supabase_client, "artist@example.com", "definitely-wrong")
    if "error" in result:
        print("PASS — wrong password correctly rejected:", result["error"])
    else:
        print("FAIL — wrong password was accepted (should never happen)")

wrong_password_test(None)

---
## What you built today

- Registered and logged in users with Supabase email/password auth
- Understood JWT structure: header, payload, signature
- Created a get_current_user() FastAPI dependency
- Protected routes with Depends() and verified 401 rejection without a token

**Next session:** Session 7 — Frontend HTML/CSS/JS — build the complete browser interface!